<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Taller 2 · De Colab a Excel
**Laboratorio Excel + Colab** — Colab prepara y valida; tú construyes el tablero en Excel · ⏱ 20 min en Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/taller_2_colab_a_excel.ipynb) &nbsp; [Ver el taller completo en el tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-2)

Este cuaderno es una parte del taller. Los pasos en Excel están en el tablero.

**Cómo usarlo:** *Archivo › Guardar una copia en Drive* → *Entorno de ejecución › Ejecutar todas*. Las celdas con listas desplegables se pueden cambiar y volver a ejecutar con ▶. No necesitas programar.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## Paso 1 · Seis tablas separadas
La empresa guarda sus datos en seis tablas unidas por llaves (ver [La empresa](https://omarli-360.github.io/excel-corporativo-12-claves/#empresa)). Una tabla dinámica, en cambio, trabaja mejor con **una sola tabla** que lo tenga todo: ciudad, vendedor, cliente, segmento, línea y producto en cada fila.

In [ ]:
# @title Ver el tamaño de cada tabla
tam = pd.Series({"ventas": len(ventas), "clientes": len(clientes), "productos": len(productos), "vendedores": len(vendedores), "sedes": len(sedes), "metas": len(metas)}).sort_values()
fig = go.Figure(go.Bar(x=tam.values, y=tam.index, orientation="h", marker_color=[AZUL if k == "ventas" else GRIS for k in tam.index],
                       text=[f"{v:,}".replace(",", ".") for v in tam.values], textposition="outside", cliponaxis=False))
fig.update_layout(title="Ventas es la tabla grande; las demás son maestros que la explican", xaxis_type="log", xaxis_title="Filas (escala logarítmica)", height=320)
fig.show()
display(Markdown("Se usa escala logarítmica porque ventas tiene miles de filas y sedes solo seis: con escala normal, los maestros no se verían."))

## Paso 2 · Unir las tablas
Colab trae a cada línea de venta los datos de sus maestros, usando las llaves:

| Dato que se trae | Desde | Llave | En Excel sería |
|---|---|---|---|
| ciudad y región | sedes | id_sede | 2 columnas con BUSCARX |
| nombre del vendedor | vendedores | id_vendedor | 1 columna con BUSCARX |
| cliente y segmento | clientes | id_cliente | 2 columnas con BUSCARX |
| producto y línea | productos | sku | 2 columnas con BUSCARX |

Además crea columnas de tiempo (año, mes, nombre del mes, trimestre). En Excel serían unas **8 columnas de fórmulas sobre 17.106 filas: más de 130.000 fórmulas**. Colab lo hace en un segundo y, sobre todo, **comprueba** el resultado.

In [ ]:
# @title Unir las tablas

MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
def tabla_plana(v):
    d = (v.merge(sedes[["id_sede", "ciudad", "region"]], on="id_sede", validate="many_to_one")
          .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
          .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
          .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
          .sort_values("id_linea"))
    return pd.DataFrame({"fecha": d.fecha.dt.date, "anio": d.fecha.dt.year, "mes": d.fecha.dt.month,
        "nombre_mes": d.fecha.dt.month.map(lambda m: MESES_ES[m - 1]), "trimestre": "T" + d.fecha.dt.quarter.astype(str),
        "id_factura": d.id_factura, "ciudad": d.ciudad, "region": d.region, "vendedor": d.nombre_vendedor,
        "cliente": d.razon_social, "segmento": d.segmento, "canal": d.canal, "categoria": d.categoria,
        "producto": d.producto, "cantidad": d.cantidad, "subtotal": d.subtotal, "margen_bruto": d.margen_bruto, "estado": d.estado})
def metas_planas(m):
    m = m.merge(sedes[["id_sede", "ciudad"]], on="id_sede")
    return pd.DataFrame({"ciudad": m.ciudad, "anio": m.anio, "mes": m.mes, "nombre_mes": m.mes.map(lambda x: MESES_ES[x - 1]),
        "fecha_mes": pd.to_datetime(dict(year=m.anio, month=m.mes, day=1)).dt.date, "meta_subtotal": m.meta_subtotal})

plana = tabla_plana(ventas)
mplana = metas_planas(metas)
display(Markdown(f"Tabla unida: **{miles(len(plana))} filas × {plana.shape[1]} columnas**."))
display(plana.head(5).style.format({"subtotal": pesos, "margen_bruto": pesos}).hide(axis="index"))

## Paso 3 · Validar antes de entregar
Cuatro pruebas. Si alguna falla, el libro no se entrega.

In [ ]:
# @title Validar la tabla unida
pruebas = [
    ("Ninguna fila perdida ni agregada", len(plana) == len(ventas)),
    ("Todas las llaves cruzaron (sin vacíos en ciudad, vendedor, cliente ni producto)", plana[["ciudad", "vendedor", "cliente", "producto"]].notna().all().all()),
    ("El total de venta cuadra con la fuente", plana["subtotal"].sum() == ventas["subtotal"].sum()),
    ("Cada sede tiene 24 metas mensuales", (mplana.groupby("ciudad").size() == 24).all()),
]
for texto, ok in pruebas:
    display(Markdown(f"{'✓' if ok else '✗'} {texto}"))
cuadre = pd.DataFrame({"tabla": ["Fuente (ventas.csv)", "Tabla unida"], "millones": [ventas.subtotal.sum() / 1e6, plana.subtotal.sum() / 1e6]})
fig = px.bar(cuadre, x="tabla", y="millones", text=cuadre.millones.map(lambda v: f"$ {v:,.0f} M".replace(",", ".")), color="tabla", color_discrete_sequence=[GRIS, AZUL])
fig.update_layout(title="Cuadre: el total de la tabla unida es idéntico al de la fuente", showlegend=False, yaxis_title=MM, xaxis_title="", height=340)
fig.show()

## Paso 4 · Exportar el libro para Excel
Se crea `Andina_listo_para_dinamicas.xlsx` con las Tablas `tVentas` y `tMetas`, una hoja `Panel` vacía y una hoja `LEEME`.

In [ ]:
# @title Crear y descargar Andina_listo_para_dinamicas.xlsx
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
wb = Workbook(); ws = wb.active; ws.title = "LEEME"; ws.column_dimensions["B"].width = 110
for r, t in enumerate(["Distribuidora Andina S.A.S. — Libro listo para tablas dinámicas (creado en Colab, Taller 2)", "",
                       "Panel → construye aquí tu tablero.", "Ventas (tVentas) → una fila por línea de factura, con todo unido.",
                       "Metas (tMetas) → meta mensual de venta neta por ciudad.", "", "Filtra estado = Anulada antes de cualquier pregunta comercial."], start=2):
    ws.cell(row=r, column=2, value=t).font = Font(name="Arial", size=14 if r == 2 else 11, bold=(r == 2), color="00006A" if r == 2 else "222222")
p = wb.create_sheet("Panel"); p.sheet_view.showGridLines = False
p["B2"] = "PANEL COMERCIAL · DISTRIBUIDORA ANDINA"; p["B2"].font = Font(name="Arial", size=18, bold=True, color="FFFFFF")
for c in "BCDEFGHIJKLMNOP": p[f"{c}2"].fill = PatternFill("solid", fgColor="00006A")
for nombre, tabla, d in [("Ventas", "tVentas", plana), ("Metas", "tMetas", mplana)]:
    h = wb.create_sheet(nombre); h.append(list(d.columns))
    for fila in d.itertuples(index=False): h.append(list(fila))
    t = Table(displayName=tabla, ref=f"A1:{get_column_letter(d.shape[1])}{len(d) + 1}"); t.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
    h.add_table(t); h.freeze_panes = "A2"
    for j, c in enumerate(d.columns, start=1):
        h.column_dimensions[get_column_letter(j)].width = min(max(len(c), int(d[c].astype(str).str.len().head(200).max())) + 2, 38)
        fmt = "#,##0" if c in ("subtotal", "margen_bruto", "meta_subtotal") else "yyyy-mm-dd" if c in ("fecha", "fecha_mes") else None
        if fmt:
            for (celda,) in h.iter_rows(min_row=2, min_col=j, max_col=j): celda.number_format = fmt
wb.save("Andina_listo_para_dinamicas.xlsx")

if EN_COLAB:
    from google.colab import files
    files.download("Andina_listo_para_dinamicas.xlsx")
    display(Markdown("Descarga iniciada: **Andina_listo_para_dinamicas.xlsx**. Si no aparece, búscalo en el panel de archivos (ícono de carpeta) › clic derecho › Descargar."))
else:
    display(Markdown("Archivo creado: **Andina_listo_para_dinamicas.xlsx** (en la carpeta del cuaderno)."))

## Paso 5 · Ahora en Excel
Abre el libro descargado y sigue la **Parte B** del taller en el tablero: hoja de cálculos, tres tablas dinámicas, gráficos dinámicos, segmentaciones, escala de tiempo e indicadores.

👉 [Taller 2 · Parte B en el tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-2)

## Resumen
* Colab une, crea columnas y valida: tareas repetitivas donde Excel es lento y propenso a errores.
* Excel recibe una sola tabla limpia, ideal para tablas dinámicas y segmentaciones.
* Un libro solo se entrega si pasa las pruebas de validación.

**Siguiente:** [Taller 3 · Tablero para el comité](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-3)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>